# Credit Card Fraud Detection & Financial Risk Analytics

## tl;dr
This notebook reproduces the verified KPIs, quality checks, segment analysis, and time trends from the supplied CSV. Run all cells after installing `requirements.txt`.

## Context & Methods

**Decision use:** monitor fraud exposure and identify segments for review.  
**Unit of analysis:** one unique `Transaction_ID`.  
**Fraud rate:** fraudulent transactions / all transactions.  
**Fraud loss:** sum of `Amount` where `Fraud_Flag = 1`.

### Key Assumptions
- `Online` is card-not-present; `POS` and `ATM` are card-present.
- Authentication is derived from the chip and PIN flags.
- Segment differences are descriptive, not causal or predictive.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data' / 'raw').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'python'))
from fraud_analysis import load_and_clean_data, build_analysis_tables
RAW_FILE = ROOT / 'data' / 'raw' / 'credit_card_fraud_2025.csv'

## Data

### 1. Load, validate, and clean

In [ ]:
cleaned, quality_report = load_and_clean_data(RAW_FILE)
quality_report

In [ ]:
pd.DataFrame({
    'Column': cleaned.columns,
    'Data_Type': cleaned.dtypes.astype(str).values,
    'Missing_Values': cleaned.isna().sum().values,
    'Distinct_Values': cleaned.nunique(dropna=False).values,
})

## Results

### 2. Calculate verified KPIs

In [ ]:
tables, kpis = build_analysis_tables(cleaned)
pd.DataFrame({'KPI': kpis.keys(), 'Value': kpis.values()})

### 3. Compare channel, card-presence, and authentication risk

In [ ]:
display(tables['fraud_by_channel'])
display(tables['fraud_by_card_presence'])
display(tables['fraud_by_authentication'])

### 4. Review geographic and merchant-category exposure

In [ ]:
display(tables['fraud_by_country'].head(10))
display(tables['fraud_by_category'].head(10))

### 5. Inspect time trends

In [ ]:
monthly = tables['monthly_fraud_trend']
ax = monthly.plot(x='Month', y='Fraud_Rate', marker='o', figsize=(11, 4.5), color='#2F6B9A', legend=False)
ax.set_title('Monthly Fraud Rate Trend')
ax.set_ylabel('Fraud rate')
ax.yaxis.set_major_formatter(lambda x, pos: f'{x:.2%}')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

### 6. Review suspicious patterns and stable-volume segment rankings

In [ ]:
display(tables['suspicious_patterns'])
display(tables['high_risk_segments'].head(15))

## Takeaways

- Resume fraud-rate and fraud-loss figures reconcile to the source rows.
- Channel and card-presence rates are very close; no material channel effect is established.
- Rankings are descriptive and should be interpreted with volume and multiple-comparison caution.
- High-value transactions matter operationally because each fraud event has larger loss impact.